In [ ]:
import os
from pathlib import Path
import logging
import shutil
import time
import gc
import glob
import re

from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType
import xarray as xr
import fsspec
import rioxarray
import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd
import holoviews as hv
import hvplot.pandas

import teehr
from teehr.evaluation.spark_session_utils import create_spark_session

LINE_PLOT_HEIGHT = 300
LINE_PLOT_WIDTH = 600

# Set global defaults for all line plots
hv.opts.defaults(
    hv.opts.Curve(
        bgcolor="#e7e9ecb8",
        show_grid=True,
        gridstyle={'grid_line_alpha': 0.5, 'grid_line_color': 'white'},
        frame_width=LINE_PLOT_WIDTH,
        frame_height=LINE_PLOT_HEIGHT
    )
)


logger = logging.getLogger(__name__)

teehr.__version__

In [ ]:
NUM_EXECUTORS = 8
NUM_CORES = 7
EXECUTOR_MEMORY = "50g"
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=NUM_EXECUTORS,
    executor_memory=EXECUTOR_MEMORY,
    executor_cores=NUM_CORES,
    aws_profile="admin-user"
)

dir_path = "/data/playground/slamont/teehr/warehouse/sedona/mat_processing/mat_warehouse"

# USE EXISTING:
ev = teehr.LocalReadWriteEvaluation(
    spark=spark,
    dir_path=dir_path,
    create_dir=False    
)

#### Add the unit and variable names for temperature to the warehouse

In [ ]:
unit_sdf = ev.units.filter("name = 'K'").to_sdf()

In [ ]:
variable_sdf = ev.variables.filter("name = 'temperature_hourly_mean'").to_sdf()

In [ ]:
%%time
ev.write.to_warehouse(
    source_data=unit_sdf,
    table_name="units",
    write_mode="append",
    catalog_name="iceberg",
    namespace_name="teehr"
)

In [ ]:
%%time
ev.write.to_warehouse(
    source_data=variable_sdf,
    table_name="variables",
    write_mode="append",
    catalog_name="iceberg",
    namespace_name="teehr"
)

#### Load the MAT timeseries

In [ ]:
mat_sdf = ev.table(table_name="primary_timeseries").to_sdf()

In [ ]:
%%time
ev.write.to_warehouse(
    source_data=mat_sdf,
    table_name="primary_timeseries",
    write_mode="append",
    catalog_name="iceberg",
    namespace_name="teehr"
)